In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [2]:

sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [3]:
vocab_size = 2000
tok  = Tokenizer(num_words = vocab_size,oov_token = "<OOV>")
tok.fit_on_texts(sentences)
seqs = tok.texts_to_sequences(sentences)
maxlen = max(len(s) for s in seqs)
X = pad_sequences(seqs,maxlen = maxlen,padding = 'post')
y = labels

# Simple Explanation: Preparing Text for a Neural Network

This code turns sentences into numbers so a neural network can understand them.

## The Problem
Computers can't read words directly — they need numbers. So we need to:
1. Turn each word into a number
2. Make sure every sentence has the same length

## Step-by-Step

**1. Pick vocabulary size**
\`\`\`python
vocab_size = 2000
\`\`\`
Only the 2000 most common words will be used. Rare words get ignored.

**2. Create the tokenizer**
\`\`\`python
tok = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
\`\`\`
This sets up a tool that will convert words → numbers.
`oov_token="<OOV>"` means: if a word isn't in our top 2000, replace it with a placeholder called `<OOV>` (Out Of Vocabulary) instead of breaking.

**3. Learn the vocabulary**
\`\`\`python
tok.fit_on_texts(sentences)
\`\`\`
The tokenizer reads all the sentences and builds a word → number dictionary. Common words get smaller numbers.

**4. Convert sentences to numbers**
\`\`\`python
seqs = tok.texts_to_sequences(sentences)
\`\`\`
Each sentence becomes a list of numbers.

Example:
`"I love pizza"` → `[5, 42, 89]`

**5. Find the longest sentence**
\`\`\`python
maxlen = max(len(s) for s in seqs)
\`\`\`
Sentences have different lengths. This finds the length of the longest one.

**6. Make all sentences the same length (padding)**
\`\`\`python
X = pad_sequences(seqs, maxlen=maxlen, padding='post')
\`\`\`
Shorter sentences get zeros added at the end until they match the longest one.

Example:
`[5, 42, 89]` → `[5, 42, 89, 0, 0]`

**7. Store the labels**
\`\`\`python
y = labels
\`\`\`
These are the correct answers (e.g. spam/not spam) for each sentence.

## In One Sentence
This code turns text into equal-length number sequences (`X`) with matching labels (`y`) so the data is ready to train a neural network.

In [4]:

X[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

In [5]:
embed_dim = 16 # har ek word ki embedding ki length kitni honi chaiye
rnn_units = 8 # for hidden layer

# Brief Explanation

\`\`\`python
embed_dim = 16
rnn_units = 8
\`\`\`

- **`embed_dim = 16`** — Each word will be represented as a vector (list) of **16 numbers**. This is the size of the "embedding" — a dense numeric representation that captures word meaning. Bigger = more expressive, but more memory/compute needed.

- **`rnn_units = 8`** — The number of neurons (units) in the RNN's hidden layer. This controls how much information the RNN can "remember" and process at each step while reading the sequence.

## In short
- `embed_dim` = how rich each word's representation is
- `rnn_units` = how much "memory/processing power" the RNN layer has

In [6]:
inp = Input(shape = (maxlen,), dtype="int32", name="input")
x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True, name="embed")(inp)
rnn = SimpleRNN(units=rnn_units, return_sequences=False, name="simple_rnn")
x_last = rnn(x)
out = Dense(1, activation="sigmoid", name="out")(x_last)
model = Model(inputs=inp, outputs=out)

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.fit(X,y, epochs=25, batch_size=8, verbose=1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5667 - loss: 0.6929
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6000 - loss: 0.6811 
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7000 - loss: 0.6706 
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8000 - loss: 0.6585 
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8333 - loss: 0.6481 
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9333 - loss: 0.6349 
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9333 - loss: 0.6224 
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9333 - loss: 0.6077 
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9667 - loss: 0.5932 
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9667 - loss: 0.5773 
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9667 - loss: 0.5600 
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9667 - loss: 0.5413 
Ep

In [8]:

intermediate_model = Model(inputs=model.inputs, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])

In [9]:
from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape=(maxlen,), dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)  # reuse trained embedding

# Create RNN with return_sequences=True
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# DO NOT CALL build() manually
seq_hidden = rnn_seq(seq_emb)  # builds automatically

# Copy trained RNN weights
try:
    trained_weights = model.get_layer('simple_rnn').get_weights()
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Inspect
idx = 0
example_seq = X[idx:idx+1]  # shape (1, maxlen)
hidden_seq = inspect_model.predict(example_seq)

print("Sentence:", sentences[idx])
print("Token ids:", example_seq)
print("Hidden states per timestep shape:", hidden_seq.shape)
print("Hidden states (timesteps x units):")
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[ 0.109 -0.052  0.012 -0.047  0.04   0.057 -0.11   0.08 ]
 [ 0.088 -0.041  0.167 -0.196  0.347  0.15  -0.111  0.076]
 [-0.071  0.107  0.438 -0.226  0.029  0.067 -0.252 -0.137]
 [-0.274  0.389  0.168 -0.28   0.195  0.074 -0.118  0.286]
 [-0.274  0.389  0.168 -0.28   0.195  0.074 -0.118  0.286]]


# Explanation: Inspecting Hidden States of a Trained RNN

This code builds a **second, smaller model** that lets you "peek inside" the RNN — seeing the hidden state at every timestep, not just the final one.

## Step-by-Step

### 1. New input for inspection
\`\`\`python
seq_inp = Input(shape=(maxlen,), dtype='int32')
\`\`\`
Creates a fresh input placeholder (same shape as before: one sequence of token IDs).

### 2. Reuse the trained word embeddings
\`\`\`python
seq_emb = model.get_layer('embed')(seq_inp)
\`\`\`
👉 **This is where the word embedding is used.**
- `model.get_layer('embed')` pulls out the **already-trained embedding layer** from your original model (the one trained with `embed_dim = 16`).
- Applying it to `seq_inp` converts each token ID into its **16-dimensional embedding vector** — using the *same* learned weights, not new random ones.
- So every word in the sentence becomes a 16-number vector here, exactly as it was during training.

### 3. Create a new RNN that returns ALL hidden states
\`\`\`python
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')
\`\`\`
- Normally an RNN only outputs the **final** hidden state.
- `return_sequences=True` makes it output the hidden state **at every single timestep** — so you can see how the RNN's "understanding" evolves word by word.
- `rnn_units = 8` → each hidden state is a vector of 8 numbers.

### 4. Build the RNN by calling it
\`\`\`python
seq_hidden = rnn_seq(seq_emb)
\`\`\`
Passes the embeddings through this new RNN layer. Keras automatically builds the layer's weight shapes here (that's why the comment says "DO NOT CALL build() manually").

### 5. Copy the trained weights into this new RNN
\`\`\`python
trained_weights = model.get_layer('simple_rnn').get_weights()
rnn_seq.set_weights(trained_weights)
\`\`\`
- This new `rnn_seq` layer starts with **random** weights by default.
- These lines copy the **actual trained weights** from your original model's RNN layer (`simple_rnn`) into this new one.
- Now `rnn_seq` behaves *exactly* like the trained RNN, but it reveals all timesteps instead of just the last one.

### 6. Build the inspection model
\`\`\`python
inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)
\`\`\`
A new mini-model: **input → embedding → RNN**, with output = hidden state at every timestep.

### 7. Run it on one example sentence
\`\`\`python
example_seq = X[idx:idx+1]
hidden_seq = inspect_model.predict(example_seq)
\`\`\`
Feeds in one sentence and gets back all its hidden states.

## Reading the Output

\`\`\`
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
\`\`\`

| Timestep | Token ID | Word    | Hidden state (8 numbers) |
|----------|----------|---------|---------------------------|
| 1        | 3        | I       | `[-0.066 -0.002  0.043  0.018  0.106  0.02  -0.035 -0.047]` |
| 2        | 26       | love    | `[ 0.26   0.114  0.001 -0.03  -0.172 -0.178  0.055  0.067]` |
| 3        | 2        | this    | `[ 0.214  0.188 -0.154  0.008  0.226  0.197  0.001  0.069]` |
| 4        | 7        | product | `[-0.187  0.12   0.062  0.36  -0.121  0.159 -0.034  0.218]` |
| 5        | 0        | *(padding)* | `[-0.187  0.12   0.062  0.36  -0.121  0.159 -0.034  0.218]` |

- Each row = the RNN's "memory" state **after reading that word**.
- Timestep 5's token ID is `0` — that's **padding**, not a real word (this matches `pad_sequences(..., padding='post')` from earlier).
- Notice timesteps 4 and 5 have the **identical** hidden state — makes sense, since padding carries no new information, so the RNN's memory doesn't change once it hits the padding zeros.

## Which Embedding Is Being Used?

The **word embedding** here is the one defined by:
\`\`\`python
embed_dim = 16
\`\`\`
from your earlier model, stored in the layer named `'embed'`. This code does **not** create a new embedding — it **reuses the already-trained one** via:
\`\`\`python
model.get_layer('embed')(seq_inp)
\`\`\`
So each word (token ID) is turned into the same 16-dimensional vector it learned during original training — this inspection model just lets you see what the RNN *does* with those embeddings, timestep by timestep.

# Problems with RNN (Brief Explanation)

### 1. Long-term memory is weak
RNNs struggle to remember information from many timesteps ago. By the time they reach the end of a long sequence, earlier words are mostly "forgotten."

### 2. Vanishing Gradient Problem
During backpropagation through time, gradients shrink exponentially as they're passed backward through many timesteps. This makes weights for early timesteps barely update — so the network fails to learn long-range patterns.

### 3. Exploding Gradient Problem
The opposite issue — gradients can grow exponentially instead of shrinking, becoming huge numbers. This makes training unstable (weights update erratically, loss can become `NaN`).

### 4. No long dependencies are getting captured
Because of vanishing gradients, RNNs can't effectively connect words/events that are far apart in a sequence (e.g., relating the first word of a paragraph to the last).

### 5. Sequential processing is slow
RNNs process one timestep at a time, in order — step 2 needs step 1's output. This means **no parallelization** across timesteps, making training slow compared to architectures like Transformers.

## In One Line
RNNs fade out old information (weak memory + vanishing/exploding gradients), can't link far-apart words, and are slow because they must process data one step at a time.